# Proteus Arc — Real EEG Experiments

**Project:** Proteus Arc — EEG-based Alzheimer's Disease Classification  
**Dataset:** OpenNeuro ds004504 (Miltiadous et al., 2023)  
**Research Question:** Can a compact 1D CNN learn useful Alzheimer's-related patterns directly from EEG signals and generalize to unseen participants better than conventional ML models using engineered features?  

**Candidates:**
- Candidate 1: Logistic Regression, SVM, Random Forest (engineered features)
- Candidate 2: Compact 1D CNN (raw EEG windows)

**Important:** This notebook does NOT fabricate data or metrics. If the dataset is unavailable, the experiment halts with a clear error.

## 1. Environment Inspection

In [1]:
import sys
import platform
print(f"Python: {sys.version}")
print(f"Platform: {platform.platform()}")

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {DEVICE}")

Python: 3.13.12 (tags/v3.13.12:1cbe481, Feb  3 2026, 18:22:25) [MSC v.1944 64 bit (AMD64)]
Platform: Windows-11-10.0.26200-SP0
PyTorch: 2.11.0+cpu
CUDA available: False
Using device: cpu


## 2. Dependency Installation

In [2]:
!pip install -q mne openneuro-py scikit-learn pandas scipy pyyaml joblib matplotlib seaborn


[notice] A new release of pip is available: 26.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 3. Repository Access

**Option A:** Clone from GitHub (update the URL to your repository).  
**Option B:** Upload the project `.zip` manually and extract it.  

After this cell, the working directory must contain `src/`, `configs/`, `scripts/`, etc.

In [3]:
import os
import sys
import zipfile

ZIP_PATH = "/content/proteus_arc.zip"
PROJECT_ROOT = "/content/proteus_arc"
required_dirs = ["src", "configs", "scripts", "tests"]

# If the project is not already extracted, extract the uploaded ZIP.
if not os.path.isdir(PROJECT_ROOT) or not all(
    os.path.isdir(os.path.join(PROJECT_ROOT, d)) for d in required_dirs
):
    if not os.path.isfile(ZIP_PATH):
        raise FileNotFoundError(
            "Project ZIP not found at /content/proteus_arc.zip. "
            "Upload proteus_arc.zip using the Colab Files panel, then rerun this cell."
        )
    os.makedirs(PROJECT_ROOT, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, "r") as archive:
        # Guard against paths escaping the extraction directory.
        base = os.path.abspath(PROJECT_ROOT)
        for member in archive.infolist():
            destination = os.path.abspath(os.path.join(PROJECT_ROOT, member.filename))
            if os.path.commonpath([base, destination]) != base:
                raise ValueError(f"Unsafe path in ZIP archive: {member.filename}")
        archive.extractall(PROJECT_ROOT)

# Find the directory containing the required project folders.
candidates = []
for root, dirs, files in os.walk(PROJECT_ROOT):
    if all(os.path.isdir(os.path.join(root, d)) for d in required_dirs):
        candidates.append(root)
        dirs[:] = []

if not candidates:
    raise FileNotFoundError(
        "Could not locate src/, configs/, scripts/, and tests/ together. "
        "Check the contents of proteus_arc.zip."
    )

PROJECT_ROOT = candidates[0]
os.chdir(PROJECT_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print("Project root:", PROJECT_ROOT)
for d in required_dirs:
    print(f"  {d}/: OK")
print("Proteus Arc project setup complete.")


FileNotFoundError: [WinError 3] The system cannot find the path specified: '/content/proteus_arc'

## 4. Dataset Access and Path Configuration

Downloads the **derivatives** subset of OpenNeuro ds004504 using `openneuro-py`.  
The derivatives contain the preprocessed `.set` files (band-pass filtered, ASR, ICA applied by the dataset authors).

**License:** CC0 (public domain).  
**Size:** ~2.5 GB for derivatives.

In [ ]:
import os
import subprocess
import sys

DATASET_PATH = "/content/ds004504"
participants_path = os.path.join(DATASET_PATH, "participants.tsv")

if not os.path.isfile(participants_path):
    print("Downloading OpenNeuro ds004504 metadata and derivatives...")
    print("The download may take a while; do not interrupt this cell.")
    os.makedirs(DATASET_PATH, exist_ok=True)

    cmd = [
        sys.executable, "-m", "openneuro", "download",
        "--dataset", "ds004504",
        "--target-dir", DATASET_PATH,
        "--include", "participants.tsv",
        "--include", "participants.json",
        "--include", "dataset_description.json",
        "--include", "derivatives/*",
    ]
    try:
        result = subprocess.run(cmd, check=True, text=True)
    except Exception as exc:
        raise RuntimeError(
            "OpenNeuro dataset download failed. Review the preceding error. "
            "Confirm the installed openneuro-py CLI syntax and dataset availability."
        ) from exc

if not os.path.isfile(participants_path):
    raise FileNotFoundError(
        f"participants.tsv was not found at {participants_path}. "
        "The dataset download did not complete successfully."
    )

derivatives_path = os.path.join(DATASET_PATH, "derivatives")
if not os.path.isdir(derivatives_path):
    raise FileNotFoundError(
        f"Expected derivatives directory was not found at {derivatives_path}. "
        "The download may not have included the EEG derivative files required by this notebook."
    )

print("Dataset metadata found:", participants_path)
print("Derivatives directory found:", derivatives_path)


## 5. Dataset Audit

In [ ]:
import pandas as pd
import yaml

participants_df = pd.read_csv(os.path.join(DATASET_PATH, 'participants.tsv'), sep='\t')
print(f'Total participants: {len(participants_df)}')
print(f'Columns: {list(participants_df.columns)}')
print(f'\nClass distribution (Group column):')
print(participants_df['Group'].value_counts())
print(f'\nGender distribution:')
print(participants_df['Gender'].value_counts())
print(f'\nAge statistics:')
print(participants_df['Age'].describe())
print(f'\nMMSE statistics by group:')
print(participants_df.groupby('Group')['MMSE'].describe())

# Validate against project config
with open('configs/data.yaml', 'r') as f:
    data_config = yaml.safe_load(f)['dataset']

expected_classes = set(data_config['classes'])
actual_classes = set(participants_df['Group'].unique())
print(f'\nExpected classes: {expected_classes}')
print(f'Actual classes:   {actual_classes}')
if expected_classes != actual_classes:
    print(f'WARNING: Class mismatch! Expected {expected_classes}, found {actual_classes}')
else:
    print('Class labels match configuration.')

# Check for missing or duplicate participant IDs
dupes = participants_df['participant_id'].duplicated().sum()
nulls = participants_df['participant_id'].isnull().sum()
print(f'\nDuplicate participant_ids: {dupes}')
print(f'Null participant_ids: {nulls}')
if dupes > 0 or nulls > 0:
    raise ValueError('Participant metadata has duplicates or nulls. Cannot proceed.')

## 6. Participant and Class-Distribution Inspection

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Class distribution
label_map = data_config['mapping']
participants_df['Diagnosis'] = participants_df['Group'].map(label_map)
sns.countplot(data=participants_df, x='Diagnosis', ax=axes[0], palette='Set2')
axes[0].set_title('Class Distribution')
axes[0].set_ylabel('Count')

# Age distribution by group
sns.boxplot(data=participants_df, x='Diagnosis', y='Age', ax=axes[1], palette='Set2')
axes[1].set_title('Age by Diagnosis')

# MMSE by group
sns.boxplot(data=participants_df, x='Diagnosis', y='MMSE', ax=axes[2], palette='Set2')
axes[2].set_title('MMSE by Diagnosis')

plt.tight_layout()
os.makedirs('outputs/figures', exist_ok=True)
plt.savefig('outputs/figures/cohort_demographics.png', dpi=150)
plt.show()
print('Saved: outputs/figures/cohort_demographics.png')

## 7. EEG Preprocessing — Load, Verify, Filter, Window

Loads each participant's preprocessed `.set` file from `derivatives/`.  
Verifies actual channel count and sampling rate per recording.  
Applies the project's bandpass filter (0.5–45 Hz) and windows into 30 s epochs with 50% overlap.

In [ ]:
import mne
import numpy as np
from src.proteus_arc.preprocessing.filtering import EEGFilter
from src.proteus_arc.preprocessing.windowing import WindowGenerator
from src.proteus_arc.preprocessing.quality_control import QualityControl

mne.set_log_level('ERROR')

with open('configs/preprocessing.yaml', 'r') as f:
    preproc_config = yaml.safe_load(f)

eeg_filter = EEGFilter(
    l_freq=preproc_config['filtering']['l_freq'],
    h_freq=preproc_config['filtering']['h_freq']
)
window_gen = WindowGenerator(
    epoch_duration_sec=preproc_config['windowing']['epoch_duration_sec'],
    overlap_ratio=preproc_config['windowing']['overlap_ratio']
)

expected_channels = data_config['channels']
expected_sfreq = data_config['sampling_rate']

all_windows = []
all_labels = []
all_groups = []
skipped = []
recording_log = []

derivatives_dir = os.path.join(DATASET_PATH, 'derivatives')
if not os.path.isdir(derivatives_dir):
    raise FileNotFoundError(
        f'Derivatives directory not found at {derivatives_dir}. '
        'Ensure the dataset download included derivatives/*.'
    )

for _, row in participants_df.iterrows():
    pid = row['participant_id']
    group = row['Group']
    label = label_map.get(group, group)

    eeg_file = os.path.join(derivatives_dir, pid, 'eeg', f'{pid}_task-eyesclosed_eeg.set')
    if not os.path.exists(eeg_file):
        skipped.append({'participant_id': pid, 'reason': 'File not found'})
        continue

    try:
        raw = mne.io.read_raw_eeglab(eeg_file, preload=True, verbose='ERROR')
    except Exception as e:
        skipped.append({'participant_id': pid, 'reason': str(e)})
        continue

    actual_sfreq = raw.info['sfreq']
    actual_channels = raw.ch_names
    duration_sec = raw.times[-1] if len(raw.times) > 0 else 0.0

    # Channel selection: pick only the expected 19 channels if present
    available = [ch for ch in expected_channels if ch in actual_channels]
    if len(available) < len(expected_channels):
        missing_ch = set(expected_channels) - set(available)
        skipped.append({'participant_id': pid, 'reason': f'Missing channels: {missing_ch}'})
        continue
    raw.pick_channels(expected_channels, ordered=True)

    # Resample if needed
    if actual_sfreq != expected_sfreq:
        raw.resample(expected_sfreq)
        actual_sfreq = expected_sfreq

    data = raw.get_data()  # (channels, samples)

    # Quality checks
    if not QualityControl.check_nan_inf(data):
        skipped.append({'participant_id': pid, 'reason': 'NaN/Inf in signal'})
        continue
    flat = QualityControl.identify_flat_channels(data)
    if flat:
        skipped.append({'participant_id': pid, 'reason': f'Flat channels: {flat}'})
        continue

    # Apply bandpass filter
    try:
        data = eeg_filter.apply(data, actual_sfreq)
    except ValueError as e:
        skipped.append({'participant_id': pid, 'reason': f'Filter error: {e}'})
        continue

    # Windowing
    metadata = {'participant_id': pid, 'label': label, 'sfreq': actual_sfreq}
    windows, win_meta = window_gen.generate_windows(data, actual_sfreq, metadata)

    if windows.shape[0] == 0:
        skipped.append({'participant_id': pid, 'reason': 'Recording too short for windowing'})
        continue

    recording_log.append({
        'participant_id': pid, 'label': label, 'sfreq': actual_sfreq,
        'n_channels': data.shape[0], 'duration_sec': duration_sec,
        'n_windows': windows.shape[0]
    })

    all_windows.append(windows)
    all_labels.extend([label] * windows.shape[0])
    all_groups.extend([pid] * windows.shape[0])

    del raw, data, windows  # free memory

if len(all_windows) == 0:
    raise RuntimeError('No valid recordings were loaded. Check the dataset and skipped list.')

X_all = np.concatenate(all_windows, axis=0)
y_all = np.array(all_labels)
groups_all = np.array(all_groups)

print(f'\n=== Data Loading Summary ===')
print(f'Total windows: {X_all.shape[0]}')
print(f'Window shape:  {X_all.shape[1:]}  (channels, samples)')
print(f'Unique participants loaded: {len(np.unique(groups_all))}')
print(f'Skipped recordings: {len(skipped)}')
print(f'Label distribution: {dict(zip(*np.unique(y_all, return_counts=True)))}')

if skipped:
    print('\nSkipped details:')
    for s in skipped:
        print(f"  {s['participant_id']}: {s['reason']}")

# Save recording log
rec_log_df = pd.DataFrame(recording_log)
os.makedirs('outputs/audit', exist_ok=True)
rec_log_df.to_csv('outputs/audit/recording_log.csv', index=False)
pd.DataFrame(skipped).to_csv('outputs/audit/skipped_recordings.csv', index=False)
print('\nSaved: outputs/audit/recording_log.csv, outputs/audit/skipped_recordings.csv')

## 8. Signal-Quality Checks

In [ ]:
# Aggregate quality stats across all loaded windows
print('Signal quality across all loaded windows:')
print(f'  All finite: {np.isfinite(X_all).all()}')
print(f'  Global min: {X_all.min():.6f}')
print(f'  Global max: {X_all.max():.6f}')
print(f'  Global std: {X_all.std():.6f}')

# Per-channel variance check
channel_vars = X_all.reshape(-1, X_all.shape[1], X_all.shape[2]).var(axis=(0, 2))
print(f'\nPer-channel variance (across all windows):')
for i, ch in enumerate(expected_channels):
    print(f'  {ch}: {channel_vars[i]:.6f}')

## 9. Window Generation Summary

Windows were generated in Cell 7. This cell provides a summary view.

In [ ]:
print(f'Window configuration:')
print(f'  Duration: {preproc_config["windowing"]["epoch_duration_sec"]} sec')
print(f'  Overlap:  {preproc_config["windowing"]["overlap_ratio"] * 100}%')
print(f'  Samples per window: {X_all.shape[2]}')
print(f'  Channels per window: {X_all.shape[1]}')

# Windows per participant
win_per_sub = pd.Series(groups_all).value_counts().sort_index()
print(f'\nWindows per participant:')
print(f'  Min: {win_per_sub.min()}, Max: {win_per_sub.max()}, Mean: {win_per_sub.mean():.1f}')

fig, ax = plt.subplots(figsize=(14, 3))
win_per_sub.plot(kind='bar', ax=ax, color='steelblue')
ax.set_title('Windows per Participant')
ax.set_ylabel('Number of Windows')
plt.tight_layout()
plt.savefig('outputs/figures/windows_per_participant.png', dpi=150)
plt.show()

## 10. Participant-Level Train / Validation / Test Splitting

Uses a two-stage `GroupShuffleSplit` to create **mutually exclusive** participant sets:  
- Train: ~60%  
- Validation: ~20%  
- Test: ~20%  

The same split is used for both Candidate 1 (baselines) and Candidate 2 (CNN).

In [ ]:
from sklearn.preprocessing import LabelEncoder
from src.proteus_arc.evaluation.participant_split import ParticipantSplitter

le = LabelEncoder()
y_encoded = le.fit_transform(y_all)
class_names = list(le.classes_)
print(f'Label encoding: {dict(zip(class_names, le.transform(class_names)))}')

RANDOM_SEED = 42

# Stage 1: Split off Test
splitter_test = ParticipantSplitter(method='group_shuffle', test_size=0.2, random_state=RANDOM_SEED)
X_temp, X_test, y_temp, y_test, g_temp, g_test = splitter_test.split(X_all, y_encoded, groups_all)

# Stage 2: Split Train / Val from remainder
splitter_val = ParticipantSplitter(method='group_shuffle', test_size=0.25, random_state=RANDOM_SEED)
X_train, X_val, y_train, y_val, g_train, g_val = splitter_val.split(X_temp, y_temp, g_temp)

# Leakage assertions
train_p = set(g_train)
val_p = set(g_val)
test_p = set(g_test)

assert train_p.isdisjoint(val_p), f'LEAKAGE: train/val overlap: {train_p & val_p}'
assert train_p.isdisjoint(test_p), f'LEAKAGE: train/test overlap: {train_p & test_p}'
assert val_p.isdisjoint(test_p), f'LEAKAGE: val/test overlap: {val_p & test_p}'

print(f'\n=== Participant-Level Split ===')
print(f'Train:      {len(train_p):>3} participants, {len(y_train):>5} windows')
print(f'Validation: {len(val_p):>3} participants, {len(y_val):>5} windows')
print(f'Test:       {len(test_p):>3} participants, {len(y_test):>5} windows')
print(f'Total:      {len(train_p)+len(val_p)+len(test_p)} participants')

# Class balance per split
for name, yy in [('Train', y_train), ('Val', y_val), ('Test', y_test)]:
    counts = dict(zip(*np.unique(yy, return_counts=True)))
    decoded = {le.inverse_transform([k])[0]: v for k, v in counts.items()}
    print(f'  {name} labels: {decoded}')

# Save split manifest
split_manifest = {
    'train_participants': sorted(list(train_p)),
    'val_participants': sorted(list(val_p)),
    'test_participants': sorted(list(test_p)),
    'random_seed': RANDOM_SEED
}
with open('outputs/audit/split_manifest.yaml', 'w') as f:
    yaml.dump(split_manifest, f)
print('\nSaved: outputs/audit/split_manifest.yaml')

## 11. Baseline Feature Extraction (Candidate 1)

In [ ]:
from src.proteus_arc.features.feature_pipeline import FeaturePipeline

feat_pipeline = FeaturePipeline(
    config_path='configs/baseline.yaml',
    sfreq=float(expected_sfreq),
    channel_names=expected_channels
)

print('Extracting features for training set...')
X_train_feat = feat_pipeline.transform(X_train).to_numpy(dtype=float)
print(f'  Train features shape: {X_train_feat.shape}')

print('Extracting features for validation set...')
X_val_feat = feat_pipeline.transform(X_val).to_numpy(dtype=float)

print('Extracting features for test set...')
X_test_feat = feat_pipeline.transform(X_test).to_numpy(dtype=float)

feature_names = list(feat_pipeline.transform(X_train[:1]).columns)
print(f'  Total features per window: {len(feature_names)}')
print(f'  Feature names (first 10): {feature_names[:10]}')

## 12. Train Logistic Regression

In [ ]:
import time
import joblib
from src.proteus_arc.models.baseline_models import BaselineModels
from src.proteus_arc.evaluation.metrics import Evaluator

os.makedirs('outputs/checkpoints', exist_ok=True)
os.makedirs('outputs/metrics', exist_ok=True)

baseline_results = {}

# Combine train+val for baselines (they don't use early stopping on val)
X_trainval_feat = np.concatenate([X_train_feat, X_val_feat], axis=0)
y_trainval = np.concatenate([y_train, y_val], axis=0)
g_trainval = np.concatenate([g_train, g_val], axis=0)

bm = BaselineModels(config_path='configs/baseline.yaml')

# --- Logistic Regression ---
lr_model = bm.get_logistic_regression()
t0 = time.time()
lr_model.fit(X_trainval_feat, y_trainval)
lr_train_time = time.time() - t0

t0 = time.time()
lr_pred = lr_model.predict(X_test_feat)
lr_proba = lr_model.predict_proba(X_test_feat)
lr_infer_time = time.time() - t0

lr_metrics = Evaluator.evaluate_model(y_test, lr_pred, lr_proba, classes=list(range(len(class_names))))
lr_metrics['Train Time (s)'] = lr_train_time
lr_metrics['Inference Time (s)'] = lr_infer_time
baseline_results['Logistic Regression'] = lr_metrics

joblib.dump(lr_model, 'outputs/checkpoints/logistic_regression.joblib')
print(f"Logistic Regression — Acc: {lr_metrics['Accuracy']:.4f}, F1: {lr_metrics['Macro F1']:.4f}, Train: {lr_train_time:.2f}s")

## 13. Train SVM

In [ ]:
svm_model = bm.get_svm()
t0 = time.time()
svm_model.fit(X_trainval_feat, y_trainval)
svm_train_time = time.time() - t0

t0 = time.time()
svm_pred = svm_model.predict(X_test_feat)
svm_proba = svm_model.predict_proba(X_test_feat)
svm_infer_time = time.time() - t0

svm_metrics = Evaluator.evaluate_model(y_test, svm_pred, svm_proba, classes=list(range(len(class_names))))
svm_metrics['Train Time (s)'] = svm_train_time
svm_metrics['Inference Time (s)'] = svm_infer_time
baseline_results['SVM'] = svm_metrics

joblib.dump(svm_model, 'outputs/checkpoints/svm.joblib')
print(f"SVM — Acc: {svm_metrics['Accuracy']:.4f}, F1: {svm_metrics['Macro F1']:.4f}, Train: {svm_train_time:.2f}s")

## 14. Train Random Forest

In [ ]:
rf_model = bm.get_random_forest()
t0 = time.time()
rf_model.fit(X_trainval_feat, y_trainval)
rf_train_time = time.time() - t0

t0 = time.time()
rf_pred = rf_model.predict(X_test_feat)
rf_proba = rf_model.predict_proba(X_test_feat)
rf_infer_time = time.time() - t0

rf_metrics = Evaluator.evaluate_model(y_test, rf_pred, rf_proba, classes=list(range(len(class_names))))
rf_metrics['Train Time (s)'] = rf_train_time
rf_metrics['Inference Time (s)'] = rf_infer_time
baseline_results['Random Forest'] = rf_metrics

joblib.dump(rf_model, 'outputs/checkpoints/random_forest.joblib')
print(f"Random Forest — Acc: {rf_metrics['Accuracy']:.4f}, F1: {rf_metrics['Macro F1']:.4f}, Train: {rf_train_time:.2f}s")

## 15. CNN Initialization (Candidate 2)

In [ ]:
from src.proteus_arc.models.cnn_1d import EEG1DCNN

with open('configs/cnn.yaml', 'r') as f:
    cnn_config = yaml.safe_load(f)

cnn_arch = cnn_config['cnn']['architecture']
cnn_train_cfg = cnn_config['cnn']['training']

# Verify architecture matches data dimensions
assert cnn_arch['in_channels'] == X_train.shape[1], \
    f"CNN expects {cnn_arch['in_channels']} channels but data has {X_train.shape[1]}"
assert cnn_arch['num_classes'] == len(class_names), \
    f"CNN expects {cnn_arch['num_classes']} classes but found {len(class_names)}"

model = EEG1DCNN(**cnn_arch)
print(model)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nTotal parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

# Smoke test
dummy = torch.randn(2, cnn_arch['in_channels'], X_train.shape[2])
out = model(dummy)
print(f'Smoke test — Input: {dummy.shape} -> Output: {out.shape}')
assert out.shape == (2, cnn_arch['num_classes'])

## 16. CNN Training and Validation

In [ ]:
from torch.utils.data import DataLoader
from src.proteus_arc.models.training import EEGDataset, CNNTrainer

torch.manual_seed(cnn_train_cfg['random_seed'])
np.random.seed(cnn_train_cfg['random_seed'])

train_dataset = EEGDataset(X_train, y_train, g_train)
val_dataset = EEGDataset(X_val, y_val, g_val)
test_dataset = EEGDataset(X_test, y_test, g_test)

train_loader = DataLoader(train_dataset, batch_size=cnn_train_cfg['batch_size'], shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=cnn_train_cfg['batch_size'], shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=cnn_train_cfg['batch_size'], shuffle=False)

# Compute class weights from TRAINING labels only
class_counts = np.bincount(y_train)
total = len(y_train)
class_weights = torch.tensor([total / max(c, 1) for c in class_counts], dtype=torch.float32)
print(f'Class weights (from train only): {class_weights}')

trainer = CNNTrainer(
    model=model,
    device=DEVICE,
    learning_rate=cnn_train_cfg['learning_rate'],
    weight_decay=cnn_train_cfg['weight_decay'],
    patience=cnn_train_cfg['early_stopping_patience'],
    class_weights=class_weights
)

cnn_save_path = 'outputs/checkpoints/best_cnn.pth'
t0 = time.time()
history = trainer.fit(train_loader, val_loader, epochs=cnn_train_cfg['epochs'], save_path=cnn_save_path)
cnn_train_time = time.time() - t0
print(f'\nTotal CNN training time: {cnn_train_time:.1f}s')

# Save training history
pd.DataFrame(history).to_csv('outputs/metrics/cnn_training_history.csv', index=False)
print('Saved: outputs/metrics/cnn_training_history.csv')

## 17. Final Held-Out Test Evaluation

Loads the **best checkpoint** (selected by validation loss) and evaluates on the held-out test participants.  
Test data was never used for training, validation, early stopping, or model selection.

In [ ]:
# Load best CNN checkpoint
model.load_state_dict(torch.load(cnn_save_path, weights_only=True, map_location=DEVICE))
model.to(DEVICE)

t0 = time.time()
cnn_test_loss, cnn_test_acc, cnn_test_f1, cnn_labels, cnn_preds, cnn_groups = trainer.val_epoch(test_loader)
cnn_infer_time = time.time() - t0

# Get probabilities for ROC-AUC
model.eval()
all_probas = []
with torch.no_grad():
    for X_batch, _, _ in test_loader:
        X_batch = X_batch.to(DEVICE)
        logits = model(X_batch)
        probas = torch.softmax(logits, dim=1)
        all_probas.append(probas.cpu().numpy())
cnn_proba = np.concatenate(all_probas, axis=0)

cnn_metrics = Evaluator.evaluate_model(cnn_labels, cnn_preds, cnn_proba, classes=list(range(len(class_names))))
cnn_metrics['Train Time (s)'] = cnn_train_time
cnn_metrics['Inference Time (s)'] = cnn_infer_time

print(f"CNN (Window-Level) — Acc: {cnn_metrics['Accuracy']:.4f}, F1: {cnn_metrics['Macro F1']:.4f}")

## 18. Participant-Level Aggregation

For each test participant, window-level predictions are aggregated via **majority voting**.

In [ ]:
all_subject_results = {}

# Baseline subject-level
for model_name, preds in [('Logistic Regression', lr_pred), ('SVM', svm_pred), ('Random Forest', rf_pred)]:
    y_true_sub, y_pred_sub = Evaluator.subject_level_vote(y_test, preds, g_test)
    sub_metrics = Evaluator.evaluate_model(y_true_sub, y_pred_sub, classes=list(range(len(class_names))))
    all_subject_results[model_name] = sub_metrics
    print(f'{model_name} (Subject-Level) — Acc: {sub_metrics["Accuracy"]:.4f}, F1: {sub_metrics["Macro F1"]:.4f}')

# CNN subject-level
y_true_sub_cnn, y_pred_sub_cnn = Evaluator.subject_level_vote(cnn_labels, cnn_preds, cnn_groups)
cnn_sub_metrics = Evaluator.evaluate_model(y_true_sub_cnn, y_pred_sub_cnn, classes=list(range(len(class_names))))
all_subject_results['1D CNN'] = cnn_sub_metrics
print(f'1D CNN (Subject-Level) — Acc: {cnn_sub_metrics["Accuracy"]:.4f}, F1: {cnn_sub_metrics["Macro F1"]:.4f}')

## 19. Metrics and Confusion Matrices

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, axes = plt.subplots(2, 2, figsize=(14, 12))
all_preds_dict = {
    'Logistic Regression': lr_pred,
    'SVM': svm_pred,
    'Random Forest': rf_pred,
    '1D CNN': cnn_preds
}
y_test_for_cm = {'Logistic Regression': y_test, 'SVM': y_test, 'Random Forest': y_test, '1D CNN': cnn_labels}

for ax, (name, preds) in zip(axes.flat, all_preds_dict.items()):
    ConfusionMatrixDisplay.from_predictions(
        y_test_for_cm[name], preds,
        display_labels=class_names,
        ax=ax, cmap='Blues', colorbar=False
    )
    ax.set_title(f'{name} (Window-Level)')

plt.tight_layout()
plt.savefig('outputs/figures/confusion_matrices.png', dpi=150)
plt.show()
print('Saved: outputs/figures/confusion_matrices.png')

## 20. Baseline vs. CNN Comparison

In [ ]:
# Window-level comparison table
all_window_metrics = {**baseline_results, '1D CNN': cnn_metrics}

comparison_cols = ['Accuracy', 'Balanced Accuracy', 'Macro F1', 'Macro Precision', 'Macro Recall', 'ROC-AUC', 'Train Time (s)', 'Inference Time (s)']
comparison_data = []
for name, m in all_window_metrics.items():
    row = {'Model': name}
    for col in comparison_cols:
        val = m.get(col, np.nan)
        row[col] = round(val, 4) if isinstance(val, float) else val
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)
print('=== Window-Level Comparison ===')
print(comparison_df.to_string(index=False))

# Subject-level comparison table
sub_comparison_data = []
for name, m in all_subject_results.items():
    row = {'Model': name}
    for col in ['Accuracy', 'Balanced Accuracy', 'Macro F1']:
        val = m.get(col, np.nan)
        row[col] = round(val, 4) if isinstance(val, float) else val
    sub_comparison_data.append(row)

sub_comparison_df = pd.DataFrame(sub_comparison_data)
print('\n=== Subject-Level Comparison ===')
print(sub_comparison_df.to_string(index=False))

# Save
comparison_df.to_csv('outputs/metrics/window_level_comparison.csv', index=False)
sub_comparison_df.to_csv('outputs/metrics/subject_level_comparison.csv', index=False)
print('\nSaved comparison tables to outputs/metrics/')

## 21. Error Analysis

In [ ]:
# Save all predictions with participant IDs for error analysis
predictions_df = pd.DataFrame({
    'participant_id': g_test,
    'y_true': y_test,
    'y_true_label': le.inverse_transform(y_test),
    'lr_pred': lr_pred, 'svm_pred': svm_pred, 'rf_pred': rf_pred
})

cnn_pred_df = pd.DataFrame({
    'participant_id': cnn_groups,
    'y_true': cnn_labels,
    'cnn_pred': cnn_preds
})

os.makedirs('outputs/predictions', exist_ok=True)
predictions_df.to_csv('outputs/predictions/baseline_predictions.csv', index=False)
cnn_pred_df.to_csv('outputs/predictions/cnn_predictions.csv', index=False)

# Identify participants all models got wrong
test_participants = sorted(set(g_test))
misclass_summary = []
for pid in test_participants:
    mask = g_test == pid
    true = y_test[mask][0]  # all same for this participant
    lr_maj = np.bincount(lr_pred[mask]).argmax()
    svm_maj = np.bincount(svm_pred[mask]).argmax()
    rf_maj = np.bincount(rf_pred[mask]).argmax()

    cnn_mask = cnn_groups == pid
    cnn_maj = np.bincount(cnn_preds[cnn_mask]).argmax() if cnn_mask.sum() > 0 else -1

    all_wrong = (lr_maj != true) and (svm_maj != true) and (rf_maj != true) and (cnn_maj != true)
    misclass_summary.append({
        'participant_id': pid,
        'true_label': le.inverse_transform([true])[0],
        'lr_pred': le.inverse_transform([lr_maj])[0],
        'svm_pred': le.inverse_transform([svm_maj])[0],
        'rf_pred': le.inverse_transform([rf_maj])[0],
        'cnn_pred': le.inverse_transform([cnn_maj])[0] if cnn_maj >= 0 else 'N/A',
        'all_wrong': all_wrong
    })

misclass_df = pd.DataFrame(misclass_summary)
print('Subject-level prediction summary:')
print(misclass_df.to_string(index=False))

hard_cases = misclass_df[misclass_df['all_wrong']]
if len(hard_cases) > 0:
    print(f'\n{len(hard_cases)} participant(s) misclassified by ALL models:')
    print(hard_cases[['participant_id', 'true_label']].to_string(index=False))
else:
    print('\nNo participants were misclassified by all models simultaneously.')

misclass_df.to_csv('outputs/predictions/subject_level_error_analysis.csv', index=False)
print('\nSaved: outputs/predictions/subject_level_error_analysis.csv')

## 22. Saving Experiment Artifacts

In [ ]:
import json

# Save experiment metadata
experiment_meta = {
    'dataset': 'OpenNeuro ds004504',
    'dataset_license': 'CC0',
    'total_participants': int(len(participants_df)),
    'loaded_participants': int(len(np.unique(groups_all))),
    'total_windows': int(X_all.shape[0]),
    'window_shape': list(X_all.shape[1:]),
    'sampling_rate': int(expected_sfreq),
    'channels': expected_channels,
    'preprocessing': preproc_config,
    'random_seed': RANDOM_SEED,
    'split': {
        'train_participants': len(train_p),
        'val_participants': len(val_p),
        'test_participants': len(test_p)
    },
    'python_version': sys.version,
    'pytorch_version': torch.__version__,
    'device': DEVICE
}

with open('outputs/experiment_metadata.json', 'w') as f:
    json.dump(experiment_meta, f, indent=2, default=str)

# Learning curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
ax1.plot(history['train_loss'], label='Train')
ax1.plot(history['val_loss'], label='Validation')
ax1.set_title('CNN Loss Curves')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.legend()

ax2.plot(history['val_f1'], label='Val F1', color='green')
ax2.set_title('CNN Validation F1')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Macro F1')
ax2.legend()
plt.tight_layout()
plt.savefig('outputs/figures/cnn_learning_curves.png', dpi=150)
plt.show()

# Zip all outputs for download
!zip -r /content/proteus_arc_outputs.zip outputs/
print('\n=== Experiment Complete ===')
print('Download outputs: /content/proteus_arc_outputs.zip')
print('\nSaved artifacts:')
for root, dirs, files in os.walk('outputs'):
    for f in files:
        print(f'  {os.path.join(root, f)}')